# Heart Disease Risk Prediction

This notebook builds a binary machine-learning classifier for the `HeartDisease` target in the supplied dataset.

> **Important:** This is an educational/research model, not a clinically validated diagnostic tool.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, brier_score_loss, confusion_matrix,
                             ConfusionMatrixDisplay, RocCurveDisplay)
import joblib

RANDOM_STATE = 42

## 1. Load and inspect the data

In [2]:
df = pd.read_csv('heart.csv')
print('Shape:', df.shape)
display(df.head())
print('Missing values:')
display(df.isna().sum().to_frame('missing'))
print('Target distribution:')
display(df['HeartDisease'].value_counts().rename_axis('class').to_frame('count'))



Shape: (918, 12)


,Age,Sex,ChestPainType,RestingBP,Cholesterol,FastingBS,RestingECG,MaxHR,ExerciseAngina,Oldpeak,ST_Slope,HeartDisease
0,40,M,ATA,140,289,0,Normal,172,N,0.0,Up,0
1,49,F,NAP,160,180,0,Normal,156,N,1.0,Flat,1
2,37,M,ATA,130,283,0,ST,98,N,0.0,Up,0
3,48,F,ASY,138,214,0,Normal,108,Y,1.5,Flat,1
4,54,M,NAP,150,195,0,Normal,122,N,0.0,Up,0


Missing values:


,missing
Age,0
Sex,0
ChestPainType,0
RestingBP,0
Cholesterol,0
FastingBS,0
RestingECG,0
MaxHR,0
ExerciseAngina,0
Oldpeak,0


Target distribution:


,count
class,
1,508
0,410


## 2. Clean invalid zero measurements

`Oldpeak = 0` is meaningful, but `RestingBP = 0` is not physiologically valid. In this dataset, zero cholesterol values are also treated as unavailable measurements. The pipeline later imputes these values using training-set medians.

In [5]:
df.loc[df['RestingBP']==0]

,Age,Sex,ChestPainType,RestingBP,Cholesterol,FastingBS,RestingECG,MaxHR,ExerciseAngina,Oldpeak,ST_Slope,HeartDisease
449,55,M,NAP,0,0,0,Normal,155,N,1.5,Flat,1


In [6]:
data2 =df.copy()
data2.loc[data2['RestingBP']==0, 'RestingBP'] =np.nan
display(data2)

,Age,Sex,ChestPainType,RestingBP,Cholesterol,FastingBS,RestingECG,MaxHR,ExerciseAngina,Oldpeak,ST_Slope,HeartDisease
0,40,M,ATA,140.0,289,0,Normal,172,N,0.0,Up,0
1,49,F,NAP,160.0,180,0,Normal,156,N,1.0,Flat,1
2,37,M,ATA,130.0,283,0,ST,98,N,0.0,Up,0
3,48,F,ASY,138.0,214,0,Normal,108,Y,1.5,Flat,1
4,54,M,NAP,150.0,195,0,Normal,122,N,0.0,Up,0
...,...,...,...,...,...,...,...,...,...,...,...,...
913,45,M,TA,110.0,264,0,Normal,132,N,1.2,Flat,1
914,68,M,ASY,144.0,193,1,Normal,141,N,3.4,Flat,1
915,57,M,ASY,130.0,131,0,Normal,115,Y,1.2,Flat,1
916,57,F,ATA,130.0,236,0,LVH,174,N,0.0,Flat,1


In [3]:
data = df.copy()
data.loc[data['RestingBP'] == 0, 'RestingBP'] = np.nan
data.loc[data['Cholesterol'] == 0, 'Cholesterol'] = np.nan

X = data.drop(columns='HeartDisease')
y = data['HeartDisease'].astype(int)

numeric_features = X.select_dtypes(include=np.number).columns.tolist()
categorical_features = X.select_dtypes(exclude=np.number).columns.tolist()
print('Numeric:', numeric_features)
print('Categorical:', categorical_features)

Numeric: ['Age', 'RestingBP', 'Cholesterol', 'FastingBS', 'MaxHR', 'Oldpeak']
Categorical: ['Sex', 'ChestPainType', 'RestingECG', 'ExerciseAngina', 'ST_Slope']


## 3. Build the preprocessing pipeline
> For more details about column transformation and pipeline [Next step ](sklearn/column.transformation.ipynb)

In [4]:
try:
    encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
except TypeError:
    encoder = OneHotEncoder(handle_unknown='ignore', sparse=False)

preprocessor = ColumnTransformer([
    ('numeric', Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
    ]), numeric_features),
    ('categorical', Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot', encoder)
    ]),  )
])

## 4. Compare candidate models using stratified cross-validation

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=2000, class_weight='balanced', random_state=RANDOM_STATE),
    'Random Forest': RandomForestClassifier(n_estimators=300, min_samples_leaf=2,
                                            class_weight='balanced', random_state=RANDOM_STATE, n_jobs=-1),
    'Gradient Boosting': GradientBoostingClassifier(random_state=RANDOM_STATE),
    'SVM (RBF)': SVC(kernel='rbf', probability=True, class_weight='balanced', random_state=RANDOM_STATE)
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
scoring = ['accuracy', 'precision', 'recall', 'f1', 'roc_auc']
rows = []

for name, classifier in models.items():
    pipeline = Pipeline([('preprocessor', preprocessor), ('classifier', classifier)])
    scores = cross_validate(pipeline, X, y, cv=cv, scoring=scoring, n_jobs=1)
    row = {'Model': name}
    for metric in scoring:
        row[metric] = scores[f'test_{metric}'].mean()
    rows.append(row)

cv_results = pd.DataFrame(rows).sort_values('roc_auc', ascending=False)
display(cv_results.style.format({m: '{:.3f}' for m in scoring}))

## 5. Train a calibrated Random Forest and evaluate on unseen test data

Calibration improves the meaning of the probability returned by `predict_proba`. The split is stratified so both classes remain proportionally represented.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=RANDOM_STATE
)

forest_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(
        n_estimators=500,
        min_samples_leaf=2,
        class_weight='balanced',
        random_state=RANDOM_STATE,
        n_jobs=-1
    ))
])

try:
    model = CalibratedClassifierCV(estimator=forest_pipeline, method='sigmoid', cv=5)
except TypeError:
    model = CalibratedClassifierCV(base_estimator=forest_pipeline, method='sigmoid', cv=5)

model.fit(X_train, y_train)
probability = model.predict_proba(X_test)[:, 1]
prediction = (probability >= 0.50).astype(int)

metrics = {
    'Accuracy': accuracy_score(y_test, prediction),
    'Precision': precision_score(y_test, prediction),
    'Recall': recall_score(y_test, prediction),
    'F1': f1_score(y_test, prediction),
    'ROC-AUC': roc_auc_score(y_test, probability),
    'Brier score': brier_score_loss(y_test, probability)
}
display(pd.Series(metrics).to_frame('value').style.format('{:.3f}'))
print('Confusion matrix:')
print(confusion_matrix(y_test, prediction))

In [ ]:
ConfusionMatrixDisplay.from_predictions(y_test, prediction)
plt.title('Confusion Matrix')
plt.show()

RocCurveDisplay.from_predictions(y_test, probability)
plt.title('ROC Curve')
plt.show()

## 6. Predict a new unseen patient record

In [ ]:
new_patient = pd.DataFrame([{
    'Age': 29,
    'Sex': 'M',
    'ChestPainType': 'ASY',
    'RestingBP': 120,
    'Cholesterol': 180,
    'FastingBS': 0,
    'RestingECG': 'Normal',
    'MaxHR': 160,
    'ExerciseAngina': 'N',
    'Oldpeak': 1.2,
    'ST_Slope': 'Flat'
}])

new_probability = float(model.predict_proba(new_patient)[0, 1])
new_class = int(new_probability >= 0.50)
print(f'Predicted class: {new_class}')
print(f'Model probability: {new_probability:.3f}')

## 7. Refit on all records and save the deployable pipeline

In [ ]:
final_model = model
final_model.fit(X, y)
joblib.dump(final_model, 'heart_disease_pipeline.joblib')
print('Saved heart_disease_pipeline.joblib')

## Interpretation

- Class `1`: the model flags possible heart-disease risk at the selected threshold.
- Class `0`: the model does not flag risk at that threshold.
- The probability is a model estimate, not a clinical diagnosis.
- Thresholds should be chosen based on the cost of false negatives versus false positives and validated on external clinical data.